# 01 Shared Feature Engineering

Build grid-level features and targets for both sub-projects.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import importlib

# Resolve project root in local or Kaggle runtime.
if Path("/kaggle/working").exists():
    ROOT = Path("/kaggle/working")
else:
    ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

if (ROOT / "src").exists():
    SRC = ROOT / "src"
else:
    SRC = Path.cwd() / "src"
if str(SRC) not in sys.path and SRC.exists():
    sys.path.append(str(SRC))

if importlib.util.find_spec("laspy") is None:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "laspy[lazrs]"])

from lidar_roraima.config import ProjectConfig
cfg = ProjectConfig.from_root(ROOT)
cfg.ensure_dirs()

# Kaggle dataset fallback path.
RAW_DATA_DIR = cfg.raw_data_dir
for candidate in [
    Path("/kaggle/input/lidar-roraima-parime-research/lidar_data"),
    Path("/kaggle/input/lidar-roraima-parime-research"),
]:
    if candidate.exists():
        RAW_DATA_DIR = candidate
        break

print("ROOT:", ROOT)
print("RAW_DATA_DIR:", RAW_DATA_DIR)
TRAIN_MAX_ROWS = None  # Set integer for constrained runtime, e.g. 1200000
cfg


In [ ]:
from lidar_roraima.features import extract_features_from_manifest, load_feature_tables

manifest = pd.read_parquet(cfg.manifests_dir / "tile_manifest.parquet")
written = extract_features_from_manifest(
    manifest=manifest,
    output_dir=cfg.features_dir,
    cell_size=10.0,
    include_targets=True,
    chunk_size=1_000_000,
    max_points_per_tile=2_000_000,  # remove cap for full local production run
)
written


In [ ]:
features = load_feature_tables(cfg.features_dir)
features.shape


In [ ]:
features.head()
